# Day 23 - Sensor Fusion: chay tren Google Colab

Notebook nay cai moi truong, tai du lieu Waymo + weights, chay `pytest` (Part E-H)
va chay ca pipeline `fusion-run-lab --fusion compare`.

**Truoc khi chay:** da commit + push phan code Part E-H len fork GitHub cua ban
(mac dinh clone tu `nvchien19/K4-Track4-Day23-Sensor-Fusion-Student`).

Chay lan luot cac cell tu tren xuong (Runtime > Run all).


## 1. Cau hinh (chinh neu can)

In [1]:
# ===== CAU HINH =====
REPO_URL = "https://github.com/nvchien19/K4-Track4-Day23-Sensor-Fusion-Student.git"
BRANCH   = "main"

# Thu muc Google Drive Day23-Fusion (Waymo/ + weights/)
DRIVE_FOLDER_ID = "1XTXA60c9gV6rkK6fWohMyHw8qpsfYBlE"

# Dat True neu du lieu da nam trong Google Drive cua ban (se mount /content/drive)
USE_DRIVE_MOUNT = False
DRIVE_DATA_DIR  = "/content/drive/MyDrive/Day23-Fusion"

# 20 = chay nhanh de test; 198 = ca segment (yeu cau khi nop bai, cham hon tren CPU)
FRAME_START = 0
FRAME_END   = 20
SEED        = 0

REPO_DIR = "/content/repo"
DATA_ROOT = "/content/day23_data"
print("Cau hinh xong.")


Cau hinh xong.


## 2. Cai thu vien phu thuoc

In [2]:
# Colab da co san torch, numpy, scipy, opencv, matplotlib, pyyaml
%pip install -q gdown "protobuf>=6.33.5,<7" easydict shapely tqdm


## 3. Clone repo

In [3]:
import os, subprocess, sys

os.makedirs(REPO_DIR, exist_ok=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo da ton tai, cap nhat...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)

print("Noi dung repo:", sorted(os.listdir(REPO_DIR)))


Noi dung repo: ['.dockerignore', '.env.example', '.git', '.github', '.gitignore', 'CHECKPOINTS.md', 'NOTICE.md', 'README.md', 'RUBRIC.md', 'RULES.md', 'SUBMISSION.md', 'data', 'docker', 'docs', 'environment.yml', 'platform', 'student', 'tools']


## 4. Dat bien moi truong

In [4]:
STUDENT_ROOT  = os.path.join(REPO_DIR, "student")
PLATFORM_ROOT = os.path.join(REPO_DIR, "platform")
WAYMO_READER  = os.path.join(PLATFORM_ROOT, "third_party", "waymo_reader")

os.environ["DAY23_STUDENT_ROOT"] = STUDENT_ROOT
os.environ["FUSION_LAB_PLATFORM"] = PLATFORM_ROOT
for _p in (PLATFORM_ROOT, WAYMO_READER):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print("Python:", sys.version.split()[0])
print("DAY23_STUDENT_ROOT =", STUDENT_ROOT)
print("FUSION_LAB_PLATFORM =", PLATFORM_ROOT)


Python: 3.13.15
DAY23_STUDENT_ROOT = /content/repo/student
FUSION_LAB_PLATFORM = /content/repo/platform


## 5. Tai du lieu Waymo + weights

In [5]:
import glob, os
from pathlib import Path

if USE_DRIVE_MOUNT:
    from google.colab import drive
    drive.mount("/content/drive")
    source = DRIVE_DATA_DIR
else:
    import gdown
    os.makedirs(DATA_ROOT, exist_ok=True)
    url = f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}"
    try:
        gdown.download_folder(id=DRIVE_FOLDER_ID, output=DATA_ROOT,
                              quiet=False, use_cookies=False)
    except TypeError:
        gdown.download_folder(url, output=DATA_ROOT, quiet=False)
    source = DATA_ROOT

segments = glob.glob(os.path.join(source, "**", "*.tfrecord"), recursive=True)
weights  = (glob.glob(os.path.join(source, "**", "*.pth"), recursive=True)
            + glob.glob(os.path.join(source, "**", "*.zip"), recursive=True))
print(f"Tim thay {len(segments)} segment, {len(weights)} file weights.")
for s in segments:
    print("  segment:", s)
for w in weights:
    print("  weights:", w)
if not segments:
    raise SystemExit("Khong tim thay .tfrecord. Kiem tra lai quyen truy cap Google Drive.")


Retrieving folder contents


Retrieving folder 1eqAu5V4GSczHmsUSlE9nZUvIWJefinR6 Waymo
Processing file 1rYJBdmlkr35_UnTRolXJtCMpMRUIZLqL training_segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord
Processing file 1XjZBZl-ekLP59tHrFiwIplRu09R7rU-M training_segment-10072231702153043603_5725_000_5745_000_with_camera_labels.tfrecord
Processing file 1vyb1IOuokToRy7PONzy7f2eXMOhCyi63 training_segment-10094743350625019937_3420_000_3440_000_with_camera_labels.tfrecord
Processing file 1aimRCubcZHQHpYlQHq6X_ZYo8RL7BA_O training_segment-10963653239323173269_1924_000_1944_000_with_camera_labels.tfrecord
Retrieving folder 1HdF8FWbOC8S1eLbDO6FI95nTOTnPKabO weights
Retrieving folder 14oTGCXNFs5TWq1MJ9Ab0UwwF4cAxFujz pretrained_darknet
Processing file 10i0Bj53uQZiwZvMv-E9dqTEWUpz1XTNB complex_yolov4_mse_loss.pth
Processing file 1JKpVKI0FJNYo1xA_RGtuKA2mrdlE3mjF complex_yolov4.cfg
Retrieving folder 1T0hVkS_jnoSHe_q9STYYFtzgrBSK4h2a pretrained_fpn-resnet
Processing file 1eG8F_EAbs6Mxq3vcDSmNLCXFKAQC8RAR fpn_r

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1rYJBdmlkr35_UnTRolXJtCMpMRUIZLqL
From (redirected): https://drive.google.com/uc?id=1rYJBdmlkr35_UnTRolXJtCMpMRUIZLqL&confirm=t&uuid=aebe44b2-5d0c-4eea-a470-5f71f94689ce
To: /content/day23_data/Waymo/training_segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord
100%|██████████| 986M/986M [00:04<00:00, 209MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1XjZBZl-ekLP59tHrFiwIplRu09R7rU-M
From (redirected): https://drive.google.com/uc?id=1XjZBZl-ekLP59tHrFiwIplRu09R7rU-M&confirm=t&uuid=fbbf1ec5-752a-4899-a904-ba568042188f
To: /content/day23_data/Waymo/training_segment-10072231702153043603_5725_000_5745_000_with_camera_labels.tfrecord
100%|██████████| 1.02G/1.02G [00:04<00:00, 220MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1vyb1IOuokToRy7PONzy7f2eXMOhCyi63
Fro

Tim thay 4 segment, 2 file weights.
  segment: /content/day23_data/Waymo/training_segment-10094743350625019937_3420_000_3440_000_with_camera_labels.tfrecord
  segment: /content/day23_data/Waymo/training_segment-10963653239323173269_1924_000_1944_000_with_camera_labels.tfrecord
  segment: /content/day23_data/Waymo/training_segment-10072231702153043603_5725_000_5745_000_with_camera_labels.tfrecord
  segment: /content/day23_data/Waymo/training_segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord
  weights: /content/day23_data/weights/pretrained_darknet/complex_yolov4_mse_loss.pth
  weights: /content/day23_data/weights/pretrained_fpn-resnet/fpn_resnet_18_epoch_300.pth



Download completed


## 6. Tao `student/config/paths.yaml`

In [6]:
import glob, os, shutil, yaml

DEFAULT_SEGMENT = ("training_segment-1005081002024129653_5313_150_5333_150"
                   "_with_camera_labels.tfrecord")

default_hits = [s for s in segments if os.path.basename(s) == DEFAULT_SEGMENT]
seg_path = default_hits[0] if default_hits else sorted(segments)[0]

# weights: uu tien dung thu muc chua dung ten file ma _resolve_weights tim
pth = next((w for w in weights if os.path.basename(w) == "fpn_resnet_18_epoch_300.pth"), None)
if pth is None:
    pth = next((w for w in weights if w.endswith(".pth")), None)

if pth is not None and os.path.basename(pth) == "fpn_resnet_18_epoch_300.pth":
    WEIGHTS_DIR = os.path.dirname(pth)
else:
    WEIGHTS_DIR = os.path.join(REPO_DIR, "data", "weights")
    dest_dir = os.path.join(WEIGHTS_DIR, "pretrained_fpn-resnet")
    os.makedirs(dest_dir, exist_ok=True)
    if pth is not None:
        dest = os.path.join(dest_dir, "fpn_resnet_18_epoch_300.pth")
        shutil.copy(pth, dest)
    else:
        import zipfile
        zips = [w for w in weights if w.endswith(".zip")]
        if zips:
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(dest_dir)

WAYMO_DIR = os.path.dirname(seg_path)
config = {
    "waymo_dir": WAYMO_DIR,
    "weights_dir": WEIGHTS_DIR,
    "segment": os.path.basename(seg_path),
    "frame_start": FRAME_START,
    "frame_end": FRAME_END,
}
config_path = os.path.join(STUDENT_ROOT, "config", "paths.yaml")
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.safe_dump(config, f)
print(open(config_path).read())


frame_end: 20
frame_start: 0
segment: training_segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord
waymo_dir: /content/day23_data/Waymo
weights_dir: /content/day23_data/weights/pretrained_fpn-resnet



## 7. Chay test Part E-H (`pytest`)

In [7]:
import os, subprocess, sys

env = dict(os.environ)
env["DAY23_STUDENT_ROOT"] = STUDENT_ROOT
env["FUSION_LAB_PLATFORM"] = PLATFORM_ROOT
env["PYTHONPATH"] = os.pathsep.join(
    [PLATFORM_ROOT, WAYMO_READER, env.get("PYTHONPATH", "")])

result = subprocess.run([sys.executable, "-m", "pytest", "student/tests", "-q"],
                        cwd=REPO_DIR, env=env)
print("pytest exit code:", result.returncode)


pytest exit code: 0


## 8. Chay pipeline Waymo (`--fusion compare`)

Luu y: detector chay tren CPU, 198 frame co the mat nhieu phut.


In [ ]:
import subprocess, sys

env["PYTHONUNBUFFERED"] = "1"


def run_lab(args, label):
    cmd = [sys.executable, "-m", "fusion_lab.scripts.run_lab"] + args
    print("=" * 70)
    print(label, "->", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True)
    print(result.stdout)
    print(f"[{label}] exit code: {result.returncode}")
    return result


# Buoc 1: chay 1 frame de bat loi nhanh (in day du traceback neu co)
diag = run_lab(["--config", "student/config/paths.yaml", "--fusion", "lidar",
                "--max-frames", "1"], "DIAGNOSTIC 1 frame lidar")

# Buoc 2: chi chay ca segment khi buoc 1 thanh cong
if diag.returncode == 0:
    run_lab(["--config", "student/config/paths.yaml", "--fusion", "compare",
             "--seed", str(SEED)], "FULL compare")
else:
    print(">>> DIAGNOSTIC THAT BAI: copy toan bo traceback o tren de duoc ho tro.")


## 9. Xem ket qua

In [9]:
import json, os

artifacts = os.path.join(STUDENT_ROOT, "artifacts")
print("Artifacts:", sorted(os.listdir(artifacts)) if os.path.isdir(artifacts) else "(chua co)")

metrics_path = os.path.join(artifacts, "metrics.json")
if os.path.isfile(metrics_path):
    metrics = json.load(open(metrics_path))
    print(json.dumps(metrics, indent=2))
    lidar = (metrics.get("tracking") or {}).get("lidar") or {}
    fused = (metrics.get("tracking") or {}).get("fused") or {}
    print()
    print("rmse lidar :", lidar.get("rmse"), "| matches:", lidar.get("matches"))
    print("rmse fused :", fused.get("rmse"), "| matches:", fused.get("matches"))
else:
    print("Chua co metrics.json - chay cell 8 truoc.")


Artifacts: ['.gitkeep', 'grade_run_lidar.log']
Chua co metrics.json - chay cell 8 truoc.


## 10. (Tuy chon) Tai ket qua ve may

Chay cell duoi de nén `student/artifacts` va tai xuong, sau do copy vao repo
cua ban va commit (khong sua tay file trong `artifacts/`).


In [10]:
import os, shutil, subprocess
from google.colab import files

src = os.path.join(STUDENT_ROOT, "artifacts")
archive = "/content/day23_artifacts"
shutil.make_archive(archive, "zip", src)
files.download(archive + ".zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>